In [ ]:
#!/usr/bin/env python
# coding: utf-8

"""
Universal Brain Fingerprint Combinatorial Engine 
-----------------------------------------------------------------------
Features included:
1. True Graph Maximum Flow calculation via networkx.
2. Exploratory Data Analysis (EDA): PCA, t-SNE, Glass Brains.
3. Full Ablation Cross-Modality: fMRI + Aperiodic, fMRI + All MEG, and Pure MEG_All_Bands
4. Full Resumable Checkpoint Manager.

"""

import os
import glob
import h5py
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, zscore
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import gc
from joblib import Parallel, delayed
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_flow
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import itertools
import warnings
from statsmodels.stats.contingency_tables import mcnemar

try:
    from nilearn import plotting
    NILEARN_AVAILABLE = True
except ImportError:
    NILEARN_AVAILABLE = False

warnings.filterwarnings('ignore')

# ==========================================
# Configuration & Constants
# ==========================================
class Config:
    FMRI_DIR = r"E:\Results\fmri_results"
    MEG_DIR = r"E:\Results\meg_results"
    
    OUTPUT_DIR = r"E:\Results\"
    EDA_DIR = os.path.join(OUTPUT_DIR, "EDA_Plots")
    
    METHODS = ['Pearson', 'GCA', 'mTE']
    
    FEATURE_STRATEGIES = [
        'Raw_Edges', 
        'Sub_Network', 
        'DP_Enhanced', 
        'PCA_Reconstructed', 
        'True_Max_Flow' 
    ]
    
    N_NODES = 166
    BASE_EDGES = N_NODES * (N_NODES - 1)
    
    PCA_COMPONENTS = 10     
    DP_PERCENTILE = 90      
    SUBNET_PERCENTILE = 80  
    
    Y_AXIS_MAX = 115.0      
    N_PERMUTATIONS = 1000   
    
    RESULTS_CSV = os.path.join(OUTPUT_DIR, "Ultimate_Fingerprint_Results.csv")
    HIT_RECORDS_H5 = os.path.join(OUTPUT_DIR, "Hit_Records_Cache.h5")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(EDA_DIR, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(Config.OUTPUT_DIR, "engine_run_v4_1.log"), encoding='utf-8'),
        logging.StreamHandler()
    ]
)

# ==========================================
# Checkpoint Manager
# ==========================================
class CheckpointManager:
    @staticmethod
    def load_completed_keys():
        if os.path.exists(Config.RESULTS_CSV):
            df = pd.read_csv(Config.RESULTS_CSV)
            keys = set(df['Strategy'] + "_" + df['Modality'] + "_" + df['Method_Combination'])
            return keys
        return set()

    @staticmethod
    def save_result_row(result_dict):
        df = pd.DataFrame([result_dict])
        if not os.path.exists(Config.RESULTS_CSV):
            df.to_csv(Config.RESULTS_CSV, index=False)
        else:
            df.to_csv(Config.RESULTS_CSV, mode='a', header=False, index=False)

    @staticmethod
    def save_hit_record(rec_key, hits):
        with h5py.File(Config.HIT_RECORDS_H5, 'a') as f:
            if rec_key in f:
                del f[rec_key]
            f.create_dataset(rec_key, data=hits)

    @staticmethod
    def load_hit_records():
        hits = {}
        if os.path.exists(Config.HIT_RECORDS_H5):
            with h5py.File(Config.HIT_RECORDS_H5, 'r') as f:
                for k in f.keys():
                    hits[k] = f[k][:]
        return hits

# ==========================================
# Data Loading
# ==========================================
class DataLoader:
    @staticmethod
    def extract_matrix(h5_path, method):
        try:
            with h5py.File(h5_path, 'r') as hf:
                if method == 'mTE' and 'results/te_weighted_adjacency_matrix' in hf:
                    mat = hf['results/te_weighted_adjacency_matrix'][:]
                elif method == 'Pearson' and 'results/results/corr_matrix' in hf:
                    mat = hf['results/results/corr_matrix'][:]
                elif method == 'GCA' and 'results/results/gca_fdr_pval' in hf:
                    pval_mat = hf['results/results/gca_fdr_pval'][:]
                    pval_mat = np.nan_to_num(pval_mat, nan=1.0)
                    mat = -np.log10(np.clip(pval_mat, 1e-15, 1.0)) 
                else:
                    return None
                
                vec = np.nan_to_num(mat[~np.eye(mat.shape[0], dtype=bool)], nan=0.0)
                return vec
        except Exception as e:
            logging.warning(f"Failed to read {os.path.basename(h5_path)}: {e}")
            return None

    @staticmethod
    def scan_directories():
        logging.info("Scanning directories and building master data structure...")
        master_data = {'fMRI': {m: {} for m in Config.METHODS}}
        
        for fpath in glob.glob(os.path.join(Config.FMRI_DIR, "*.h5")):
            parts = os.path.basename(fpath).replace('.h5', '').split('_')
            if len(parts) < 6: continue
            method, subj = parts[0], parts[1]
            if method not in Config.METHODS: continue
            
            vec = DataLoader.extract_matrix(fpath, method)
            if vec is not None:
                master_data['fMRI'][method].setdefault(subj, []).append(vec)

        for fpath in glob.glob(os.path.join(Config.MEG_DIR, "*.h5")):
            parts = os.path.basename(fpath).replace('.h5', '').split('_')
            if len(parts) < 6: continue
            method, subj = parts[0], parts[1]
            
            # Robust band extraction
            task_str = "_".join(parts[2:])
            band = "Wide-band"
            for b in ["Alpha", "Beta", "Delta", "Theta", "Low-Gamma", "High-Gamma", "Low-Frequency", "Wide-band", "Aperiodic-VMD", "Aperiodic"]:
                if b in task_str:
                    band = b if b != "Aperiodic" else "Aperiodic-VMD"
                    break
                    
            if method not in Config.METHODS: continue
            
            meg_key = f"MEG_{band}"
            if meg_key not in master_data:
                master_data[meg_key] = {m: {} for m in Config.METHODS}
            
            vec = DataLoader.extract_matrix(fpath, method)
            if vec is not None:
                master_data[meg_key][method].setdefault(subj, []).append(vec)
                
        return master_data

# ==========================================
# Feature Engineering Module
# ==========================================
class FeatureEngineer:
    @staticmethod
    def calculate_true_max_flow(vec):
        n_blocks = len(vec) // Config.BASE_EDGES
        flows = []
        for i in range(n_blocks):
            sub_vec = vec[i * Config.BASE_EDGES : (i + 1) * Config.BASE_EDGES]
            mat = np.zeros((Config.N_NODES, Config.N_NODES))
            mat[~np.eye(Config.N_NODES, dtype=bool)] = np.abs(sub_vec) 
            
            SCALE_FACTOR = 100000
            mat_int = (mat * SCALE_FACTOR).astype(np.int32)
            sparse_graph = csr_matrix(mat_int)
            flow_mat = np.zeros((Config.N_NODES, Config.N_NODES))
            
            for u in range(Config.N_NODES):
                for v in range(Config.N_NODES):
                    if u != v:
                        res = maximum_flow(sparse_graph, u, v)
                        flow_mat[u, v] = res.flow_value / SCALE_FACTOR
                        
            flows.append(flow_mat[~np.eye(Config.N_NODES, dtype=bool)])
        return np.concatenate(flows)

    @staticmethod
    def apply_strategy(t_list, db_list, strategy):
        if strategy == 'Raw_Edges':
            return t_list, db_list
            
        elif strategy == 'DP_Enhanced':
            db_arr = np.array(db_list)
            edge_vars = np.var(db_arr, axis=0)
            threshold = np.percentile(edge_vars, Config.DP_PERCENTILE)
            mask = edge_vars >= threshold
            return [t[mask] for t in t_list], [db[mask] for db in db_list]
            
        elif strategy == 'Sub_Network':
            db_arr = np.array(db_list)
            mean_edges = np.mean(np.abs(db_arr), axis=0)
            threshold = np.percentile(mean_edges, Config.SUBNET_PERCENTILE)
            mask = mean_edges >= threshold
            return [t[mask] for t in t_list], [db[mask] for db in db_list]
            
        elif strategy == 'PCA_Reconstructed':
            db_arr = np.array(db_list)
            t_arr = np.array(t_list)
            n_comp = min(Config.PCA_COMPONENTS, db_arr.shape[0])
            pca = PCA(n_components=n_comp)
            pca.fit(db_arr)
            db_recon = pca.inverse_transform(pca.transform(db_arr))
            t_recon = pca.inverse_transform(pca.transform(t_arr))
            return list(t_recon), list(db_recon)
            
        elif strategy == 'True_Max_Flow':
            gc.collect() 
            t_res = Parallel(n_jobs=4)(delayed(FeatureEngineer.calculate_true_max_flow)(t) for t in t_list)
            gc.collect() 
            db_res = Parallel(n_jobs=4)(delayed(FeatureEngineer.calculate_true_max_flow)(db) for db in db_list)
            return list(t_res), list(db_res)
            
        return t_list, db_list

# ==========================================
# Core Engine & Statistical Tests
# ==========================================
class FingerprintEngine:
    @staticmethod
    def evaluate(target_vectors, db_vectors):
        n_sub = len(target_vectors)
        if n_sub < 2: return 0.0, [], 1.0
        
        id_mat = np.zeros((n_sub, n_sub))
        for i in range(n_sub):
            for j in range(n_sub):
                if np.std(target_vectors[i]) == 0 or np.std(db_vectors[j]) == 0:
                    id_mat[i, j] = 0.0
                else:
                    corr, _ = pearsonr(target_vectors[i], db_vectors[j])
                    id_mat[i, j] = np.nan_to_num(corr, nan=0.0)
        
        predicted_identities = np.argmax(id_mat, axis=1)
        true_identities = np.arange(n_sub)
        correct_hits = (predicted_identities == true_identities)
        success_rate = (np.sum(correct_hits) / n_sub) * 100.0
        
        null_hits = np.zeros(Config.N_PERMUTATIONS)
        for p in range(Config.N_PERMUTATIONS):
            shuffled_labels = np.random.permutation(true_identities)
            null_hits[p] = np.sum(predicted_identities == shuffled_labels)
            
        p_val_base = (np.sum(null_hits >= np.sum(correct_hits)) + 1) / (Config.N_PERMUTATIONS + 1)
        return success_rate, correct_hits, p_val_base

    @staticmethod
    def build_fusion_vectors(modality_data, methods_to_fuse, subject_list):
        t_list, db_list = [], []
        for subj in subject_list:
            subj_t_pieces, subj_db_pieces = [], []
            for method in methods_to_fuse:
                runs = modality_data[method][subj]
                t_vec = runs[0]
                db_vec = np.mean(runs[1:], axis=0) if len(runs) > 1 else runs[0]
                
                subj_t_pieces.append(np.nan_to_num(zscore(t_vec), nan=0.0))
                subj_db_pieces.append(np.nan_to_num(zscore(db_vec), nan=0.0))
                
            t_list.append(np.concatenate(subj_t_pieces))
            db_list.append(np.concatenate(subj_db_pieces))
        return t_list, db_list

# ==========================================
# Main Execution Pipeline
# ==========================================
def main():
    logging.info("=========================================================================")
    logging.info("Initiating Ultimate Engine   (Full Ablation + Pure MEG_All_Bands Support)")
    logging.info("=========================================================================")
    
    master_data = DataLoader.scan_directories()
    completed_keys = CheckpointManager.load_completed_keys()
    
    valid_subjs = None
    for mod in master_data.keys():
        for meth in Config.METHODS:
            subjs_with_runs = set(s for s, runs in master_data[mod][meth].items() if len(runs) >= 2)
            if valid_subjs is None: valid_subjs = subjs_with_runs
            else: valid_subjs &= subjs_with_runs
                
    final_subjs = sorted(list(valid_subjs))
    if len(final_subjs) < 2:
        logging.error("Insufficient subjects for cross-validation.")
        return

    method_combinations = []
    for r in range(1, len(Config.METHODS) + 1):
        method_combinations.extend(list(itertools.combinations(Config.METHODS, r)))

    for strategy in Config.FEATURE_STRATEGIES:
        logging.info(f"\n>>> Preprocessing Strategy: {strategy} <<<")
        
        # Intra-Modality (Single Bands)
        for mod_name, mod_data in master_data.items():
            for meth_combo in method_combinations:
                combo_name = "+".join(meth_combo)
                rec_key = f"{strategy}_{mod_name}_{combo_name}"
                
                if rec_key in completed_keys: continue
                
                t_raw, db_raw = FingerprintEngine.build_fusion_vectors(mod_data, meth_combo, final_subjs)
                t_eng, db_eng = FeatureEngineer.apply_strategy(t_raw, db_raw, strategy)
                sr, correct_hits, p_val = FingerprintEngine.evaluate(t_eng, db_eng)
                
                CheckpointManager.save_hit_record(rec_key, correct_hits)
                result_dict = {
                    'Strategy': strategy, 'Modality': mod_name,
                    'Method_Combination': combo_name, 'Fusion_Level': f"{len(meth_combo)}-Method",
                    'Success_Rate': sr, 'Permutation_P_Value': p_val
                }
                CheckpointManager.save_result_row(result_dict)
                logging.info(f"[{strategy}] [{mod_name}] {combo_name}: {sr:.1f}%")

        # Cross-Modality (The Ablation Studio)
        cross_modalities_to_test = []
        
        if 'MEG_Aperiodic-VMD' in master_data:
            cross_modalities_to_test.append(('fMRI + MEG_Aperiodic-VMD', ['MEG_Aperiodic-VMD']))
            
        meg_all_bands = [k for k in master_data.keys() if k.startswith('MEG_')]
        if len(meg_all_bands) > 1:
            cross_modalities_to_test.append(('MEG_All_Bands', meg_all_bands))         # Pure MEG Fusion
            cross_modalities_to_test.append(('fMRI + MEG_All_Bands', meg_all_bands))  # Ultimate Fusion
            
        for cross_mod_name, target_meg_bands in cross_modalities_to_test:
            for meth_combo in method_combinations:
                combo_name = "+".join(meth_combo)
                rec_key = f"{strategy}_{cross_mod_name}_{combo_name}"
                
                if rec_key in completed_keys: continue
                
                includes_fmri = 'fMRI' in cross_mod_name
                
                # Handle fMRI logic if present
                if includes_fmri:
                    t_f_raw, db_f_raw = FingerprintEngine.build_fusion_vectors(master_data['fMRI'], meth_combo, final_subjs)
                    t_f_eng, db_f_eng = FeatureEngineer.apply_strategy(t_f_raw, db_f_raw, strategy)
                
                # Handle MEG Bands integration
                t_m_eng_list = [[] for _ in range(len(final_subjs))]
                db_m_eng_list = [[] for _ in range(len(final_subjs))]
                
                for band in target_meg_bands:
                    t_b_raw, db_b_raw = FingerprintEngine.build_fusion_vectors(master_data[band], meth_combo, final_subjs)
                    t_b_eng, db_b_eng = FeatureEngineer.apply_strategy(t_b_raw, db_b_raw, strategy)
                    for i in range(len(final_subjs)):
                        t_m_eng_list[i].append(t_b_eng[i])
                        db_m_eng_list[i].append(db_b_eng[i])
                        
                t_m_eng = [np.concatenate(t_m_eng_list[i]) for i in range(len(final_subjs))]
                db_m_eng = [np.concatenate(db_m_eng_list[i]) for i in range(len(final_subjs))]
                
                # Dynamic Final Concatenation
                t_multi = [np.concatenate([t_f_eng[i], t_m_eng[i]]) if includes_fmri else t_m_eng[i] for i in range(len(final_subjs))]
                db_multi = [np.concatenate([db_f_eng[i], db_m_eng[i]]) if includes_fmri else db_m_eng[i] for i in range(len(final_subjs))]
                
                sr, correct_hits, p_val = FingerprintEngine.evaluate(t_multi, db_multi)
                
                CheckpointManager.save_hit_record(rec_key, correct_hits)
                result_dict = {
                    'Strategy': strategy, 'Modality': cross_mod_name,
                    'Method_Combination': combo_name, 'Fusion_Level': f"Cross-Modal {len(meth_combo)}-Method",
                    'Success_Rate': sr, 'Permutation_P_Value': p_val
                }
                CheckpointManager.save_result_row(result_dict)
                logging.info(f"[{strategy}] [{cross_mod_name}] {combo_name}: {sr:.1f}%")
    
    logging.info("Pipeline Execution Complete.")

if __name__ == "__main__":
    main()